# Task 4 Details

## Objectives of the task
- convert continuos FICO scores into discrete bucket scores (quantization/discretization)
- main task would be to determine the range of the buckets w.r.t. to the values of the FICO scores in the provided data
- we need to find bucket boundaries that
  - summarize the data well
  - predict the behaviour
  - minimize information loss

## Method implemented
- we use OptBinning from OptimalBinning package
- **What is Optimal Binning?
  - it is a technique that aims to bucket data such that information value (IV) is maximised
  - Unlike traditional binning methods, optimal binning considers the relationship between the numerical variable and the target variable to produce more informative bins.

**As an experiment, i have also implemented manual binning method to compare analysis w.r.t. to OptBinning library**

In [1]:
%pip install optbinning
%pip install --upgrade optbinning
%pip install "scikit-learn<1.6.0"
#import neccessary libraries
import pandas as pd
import numpy as np
from optbinning import OptimalBinning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 8.6 MB/s  0:00:01m 8.1 MB/s eta 0:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2
Note: you may need to restart the kernel to use updated packages.
  Using cached optbinning-0.21.0-py3-none-any.whl.metadata (2.1 kB)
INFO: pip is looking at multiple versions of optbinning to determine which version is compatible with other requirements. This could take a while.
  Using cached optbinning-0.20.1-py3-none-any.whl.metadata (2.1 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached scikit_learn-1.5.2-cp313-cp313-macosx_12_0_arm64.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp313-cp313-macosx_12_0_arm64.whl (11.0 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstal

In [3]:
df = pd.read_csv('loan_data.csv')
df

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0
...,...,...,...,...,...,...,...,...
9995,3972488,0,3033.647103,2553.733144,42691.62787,5,697,0
9996,6184073,1,4146.239304,5458.163525,79969.50521,8,615,0
9997,6694516,2,3088.223727,4813.090925,38192.67591,5,596,0
9998,3942961,0,3288.901666,1043.099660,50929.37206,2,647,0


In [4]:
#initalize variables 
X = df['fico_score'].values
y = df['default'].values

In [5]:
# initalize the OptBinning model
optb = optb = OptimalBinning(name="fico_score", dtype="numerical", solver="cp", monotonic_trend="descending")
optb.fit(X,y)

OptimalBinning(monotonic_trend='descending', name='fico_score')

In [6]:
#putting the splits and boundaries
splits = optb.splits 
boundaries = [-np.inf] + list(splits) + [np.inf]

ratings = list(reversed(range(1, len(boundaries))))

In [7]:
#cutting neccessary columns and presenting FICO analysis results
df["optbin_rating"] = pd.cut(df["fico_score"], bins=boundaries, labels=ratings)

optbin_summary = df.groupby("optbin_rating").agg({

    "fico_score": ["min", "max", "count"],

    "default": "mean"

})

optbin_summary.columns = [

    "Min_FICO",

    "Max_FICO",

    "Count",

    "Default_Rate"

]

print(optbin_summary)

               Min_FICO  Max_FICO  Count  Default_Rate
optbin_rating                                         
11                  408       550    758      0.542216
10                  551       580    950      0.341053
9                   581       591    510      0.256863
8                   592       610    983      0.240081
7                   611       640   1945      0.172751
6                   641       649    588      0.136054
5                   650       659    658      0.110942
4                   660       672    772      0.102332
3                   673       696   1179      0.088210
2                   697       719    805      0.063354
1                   720       850    852      0.030516


/var/folders/ll/l0t2714j2sddm7tb5lxttf4m0000gn/T/ipykernel_73330/2330636141.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  optbin_summary = df.groupby("optbin_rating").agg({


In [8]:
#using the manual binning method
df = pd.read_csv("loan_data.csv")

df = df.sort_values("fico_score")

n_buckets = 5

bucket_size = len(df) // n_buckets

boundaries = []

for i in range(1, n_buckets):
    boundary = df.iloc[i * bucket_size]["fico_score"]
    boundaries.append(boundary)

bins = [-np.inf] + boundaries + [np.inf]

ratings = list(reversed(range(1, n_buckets + 1)))

df["manual_rating"] = pd.cut(
    df["fico_score"],
    bins=bins,
    labels=ratings
)

manual_summary = df.groupby("manual_rating").agg({
    "fico_score": ["min", "max", "count"],
    "default": "mean"
})

manual_summary.columns = [
    "Min_FICO",
    "Max_FICO",
    "Count",
    "Default_Rate"
]

print(manual_summary)

               Min_FICO  Max_FICO  Count  Default_Rate
manual_rating                                         
5                   408       587   2050      0.398537
4                   588       623   1971      0.215627
3                   624       653   1989      0.151332
2                   654       688   1997      0.100150
1                   689       850   1993      0.054190


/var/folders/ll/l0t2714j2sddm7tb5lxttf4m0000gn/T/ipykernel_73330/2303899973.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  manual_summary = df.groupby("manual_rating").agg({
